In [2]:
import numpy as np
import pandas as pd

import os

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/competitions/ieee-fraud-detection/sample_submission.csv
/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv
/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv


In [3]:
!pip install dagshub mlflow optuna -q

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import mlflow
import mlflow.sklearn
import dagshub
from sklearn.linear_model import LogisticRegression, RidgeClassifier, SGDClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import VarianceThreshold
from kaggle_secrets import UserSecretsClient
import warnings

warnings.filterwarnings('ignore')

dagshub.init(
    repo_owner="sophyrise",
    repo_name='ieee-cis-fraud-detection',
    mlflow=True
)

mlflow.set_experiment("LogisticRegression-Training")
print("✅ MLflow tracking URI:", mlflow.get_tracking_uri())

Accessing as sophyrise

Initialized MLflow to track repo "sophyrise/ieee-cis-fraud-detection"

Repository sophyrise/ieee-cis-fraud-detection initialized!

✅ MLflow tracking URI: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow


# **Cleaning**

In [8]:
with mlflow.start_run(run_name="LogisticRegression-Cleaning"):
    # ჩამოვტვირთოთ csv ფაილები
    train_txn = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv')
    train_id = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv')
    test_txn = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv')
    test_id = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv')

    # გავაერთიანოთ ტრანზაქციების და იდენტობის დატა ერთი სახელის ქვეშ
    train = train_txn.merge(train_id, on='TransactionID', how='left')
    test = test_txn.merge(test_id, on='TransactionID', how='left')

    print("Train shape: ", train.shape)
    print("Test shape: ", test.shape)
    print("Fraud rate: ", round(train['isFraud'].mean(), 4))

    mlflow.log_param("train_rows", train.shape[0])
    mlflow.log_param("train_cols", train.shape[1])
    mlflow.log_param("fraud_rate", round(train['isFraud'].mean(), 4))

    # 1. ამოვიღოთ სვეტები, სადაც დატას 80%-ზე მეტია ცარიელი
    missing = train.isnull().mean()
    drop_cols = missing[missing > 0.8].index.tolist()
    train_clean = train.drop(columns=drop_cols)
    test_clean = test.drop(columns=[c for c in drop_cols if c in test.columns])

    mlflow.log_param("missing_threshold", "80%")
    mlflow.log_param("cols_dropped_high_missing", len(drop_cols))

    # 2. გადავიყვანოთ კატეგორიული ცვლადები რიცხვით ფორმატში
    common_cols = list(set(train_clean.columns) & set(test_clean.columns))
    cat_cols = [c for c in common_cols
                if train_clean[c].dtype == 'object'
                and c not in ['TransactionID', 'isFraud']]

    le = LabelEncoder()
    for col in cat_cols:
        train_clean[col] = train_clean[col].astype(str)
        test_clean[col] = test_clean[col].astype(str)
        combined = pd.concat([train_clean[col], test_clean[col]])
        le.fit(combined)
        train_clean[col] = le.transform(train_clean[col])
        test_clean[col] = le.transform(test_clean[col])

    mlflow.log_param("categorical_encoding", "LabelEncoder")
    mlflow.log_param("categorical_cols", len(cat_cols))

    # 3. შევცვალოთ NaN-ები მედიანით
    num_cols = [c for c in train_clean.columns
                if train_clean[c].dtype != 'object'
                and c not in ['TransactionID', 'isFraud']]
    common_cols = [c for c in num_cols if c in test_clean.columns]

    for col in common_cols:
        median = train_clean[col].median()
        train_clean[col].fillna(median, inplace=True)
        test_clean[col].fillna(median, inplace=True)

    mlflow.log_param("numerical_imputation", "median")
    mlflow.log_param("train_shape_after_cleaning", str(train_clean.shape))

    print("✅ Cleaning done. Shape:", train_clean.shape)
    print(f"   Dropped {len(drop_cols)} high-missing columns")
    print(f"   Encoded {len(cat_cols)} categorical columns")
    print(f"   Filled  {len(num_cols)} numerical columns with median")

Train shape:  (590540, 434)
Test shape:  (506691, 433)
Fraud rate:  0.035
✅ Cleaning done. Shape: (590540, 360)
   Dropped 74 high-missing columns
   Encoded 16 categorical columns
   Filled  348 numerical columns with median
🏃 View run LogisticRegression-Cleaning at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/1/runs/3a02539e5f064ffba59a07da4d451da6
🧪 View experiment at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/1


# **Feature Engineering**

In [ ]:
with mlflow.start_run(run_name="LogisticRegression_FeatureEngineering"):
    from sklearn.preprocessing import LabelEncoder

    X_train_fe = X_train_clean.copy()
    X_test_fe = X_test_clean.copy()
    y_fe = y_train_clean.copy()

    # making numbers less extreme so the model can learn better
    X_train_fe["TransactionAmt_log"] = np.log1p(X_train_fe["TransactionAmt"])
    X_test_fe["TransactionAmt_log"] = np.log1p(X_test_fe["TransactionAmt"])

    # extract cents. fraud often has patterns in decimals so model can recognize it easily
    X_train_fe["TransactionAmt_cents"] = (X_train_fe["TransactionAmt"] % 1).round(3)
    X_test_fe["TransactionAmt_cents"] = (X_test_fe["TransactionAmt"] % 1).round(3)

    # humans often make transactions with round numbers, otherwise it is suspicious, so we create column with 0/1 values, where with 0 is defined as more likely fraud case
    X_train_fe["TransactionAmt_isround"] = (X_train_fe["TransactionAmt_cents"] == 0).astype(np.int8)
    X_test_fe["TransactionAmt_isround"] = (X_test_fe["TransactionAmt_cents"] == 0).astype(np.int8)

    # takes hour
    X_train_fe["hour"] = (X_train_fe["TransactionDT"] // 3600) % 24
    X_test_fe["hour"] = (X_test_fe["TransactionDT"] // 3600) % 24

    # takes day
    X_train_fe["day"] = (X_train_fe["TransactionDT"] // (3600 * 24)) % 7
    X_test_fe["day"] = (X_test_fe["TransactionDT"] // (3600 * 24)) % 7

    # maps time to circle thorugh sin and cos
    for col, period in [("hour", 24), ("day", 7)]:
        X_train_fe[f"{col}_sin"] = np.sin(2 * np.pi * X_train_fe[col] / period)
        X_train_fe[f"{col}_cos"] = np.cos(2 * np.pi * X_train_fe[col] / period)
        X_test_fe[f"{col}_sin"] = np.sin(2 * np.pi * X_test_fe[col] / period)
        X_test_fe[f"{col}_cos"] = np.cos(2 * np.pi * X_test_fe[col] / period)

    # drop original columns cause we created new, more important ones using them
    X_train_fe = X_train_fe.drop(columns=["TransactionDT", "hour", "day"])
    X_test_fe = X_test_fe.drop(columns=["TransactionDT", "hour", "day"])

    # break down email into two parts - provider and suffix
    for col in ["P_emaildomain", "R_emaildomain"]:
        if col in X_train_fe.columns:
            for df in [X_train_fe, X_test_fe]:
                df[f"{col}_provider"] = df[col].str.split(".").str[0]
                df[f"{col}_suffix"] = df[col].str.split(".").str[-1]

    # put card1 and card2 together to identify the culprit better
    if all(c in X_train_fe.columns for c in ["card1", "card2"]):
        X_train_fe["card1_card2"] = (
                X_train_fe["card1"].astype(str) + "_" + X_train_fe["card2"].astype(str)
        )
        X_test_fe["card1_card2"] = (
                X_test_fe["card1"].astype(str) + "_" + X_test_fe["card2"].astype(str)
        )

    # identify unusual transactions through card1
    if "card1" in X_train_fe.columns:
        card1_stats = (
            X_train_fe.groupby("card1")["TransactionAmt"]
            .agg(card1_amt_mean="mean", card1_amt_std="std", card1_count="count")
            .reset_index()
        )
        X_train_fe = X_train_fe.merge(card1_stats, on="card1", how="left")
        X_test_fe = X_test_fe.merge(card1_stats, on="card1", how="left")

        X_train_fe["card1_amt_zscore"] = (
                (X_train_fe["TransactionAmt"] - X_train_fe["card1_amt_mean"])
                / (X_train_fe["card1_amt_std"] + 1e-5)
        )
        X_test_fe["card1_amt_zscore"] = (
                (X_test_fe["TransactionAmt"] - X_test_fe["card1_amt_mean"])
                / (X_test_fe["card1_amt_std"] + 1e-5)
        )

    # loops over columns and checks if columns contain NaN, if so, it created flags(0/1)
    # so it shows 1 when the value is missing
    cols_with_missing = [c for c in X_train_fe.columns if X_train_fe[c].isnull().any()]
    for col in cols_with_missing:
        X_train_fe[f"{col}_isnan"] = X_train_fe[col].isnull().astype(np.int8)
        X_test_fe[f"{col}_isnan"] = X_test_fe[col].isnull().astype(np.int8)

    # WOE ENCODING - defines number that is connected to target without the need for many categories
    category_cols = X_train_fe.select_dtypes(include=["object"]).columns.tolist()
    HIGH_CARD_THRESHOLD = 6
    high_cardinality_cols = [c for c in category_cols if X_train_fe[c].nunique() > HIGH_CARD_THRESHOLD]
    low_cardinality_cols = [c for c in category_cols if X_train_fe[c].nunique() <= HIGH_CARD_THRESHOLD]

    woe_mappings = {}
    iv_values = {}

    for col in high_cardinality_cols:
        tmp = pd.DataFrame({"col": X_train_fe[col].astype(str),
                            "target": y_fe.values})
        grp = tmp.groupby("col")["target"].agg(["count", "mean"])
        grp["n_pos"] = grp["mean"] * grp["count"]
        grp["n_neg"] = grp["count"] - grp["n_pos"]
        total_pos = grp["n_pos"].sum()
        total_neg = grp["n_neg"].sum()
        grp["p_pos"] = grp["n_pos"] / (total_pos + 1e-8)
        grp["p_neg"] = grp["n_neg"] / (total_neg + 1e-8)
        grp["woe"] = np.log((grp["p_pos"] + 1e-8) / (grp["p_neg"] + 1e-8))
        grp["iv"] = (grp["p_pos"] - grp["p_neg"]) * grp["woe"]

        woe_mappings[col] = grp["woe"].to_dict()
        iv_values[col] = grp["iv"].sum()

        X_train_fe[f"{col}_woe"] = X_train_fe[col].astype(str).map(woe_mappings[col]).fillna(0)
        X_test_fe[f"{col}_woe"] = X_test_fe[col].astype(str).map(woe_mappings[col]).fillna(0)

    # ორიგინალი high-card სვეტები ამოვიღოთ
    X_train_fe = X_train_fe.drop(columns=high_cardinality_cols, errors="ignore")
    X_test_fe = X_test_fe.drop(columns=high_cardinality_cols, errors="ignore")

    # ── 8. ONE-HOT ENCODING — LOW CARDINALITY CATEGORICALS ───────────────────
    X_train_fe = pd.get_dummies(X_train_fe, columns=low_cardinality_cols, drop_first=True, dummy_na=True)
    X_test_fe = pd.get_dummies(X_test_fe, columns=low_cardinality_cols, drop_first=True, dummy_na=True)

    # სვეტების სინქრონიზაცია
    X_train_fe, X_test_fe = X_train_fe.align(X_test_fe, join="left", axis=1, fill_value=0)

    # ── 9. MEDIAN IMPUTATION FOR REMAINING NULLS ──────────────────────────────
    num_cols = X_train_fe.select_dtypes(include=[np.number]).columns
    medians = X_train_fe[num_cols].median()
    X_train_fe[num_cols] = X_train_fe[num_cols].fillna(medians)
    X_test_fe[num_cols] = X_test_fe[num_cols].fillna(medians)

    # just log everything
    mlflow.log_param("high_card_threshold", HIGH_CARD_THRESHOLD)
    mlflow.log_param("encoding_high_card", "WOE")
    mlflow.log_param("encoding_low_card", "OneHot")
    mlflow.log_param("amt_log_transform", True)
    mlflow.log_param("cyclic_time_encoding", True)
    mlflow.log_param("missing_indicator_flags", len(cols_with_missing))
    mlflow.log_param("woe_encoded_cols", len(high_cardinality_cols))
    mlflow.log_param("onehot_encoded_cols", len(low_cardinality_cols))
    mlflow.log_metric("features_before_fe", int(X_train_clean.shape[1]))
    mlflow.log_metric("features_after_fe", int(X_train_fe.shape[1]))
    mlflow.log_metric("new_features_created", int(X_train_fe.shape[1] - X_train_clean.shape[1]))

    import matplotlib.pyplot as plt
    import seaborn as sns

    # IV table picture drawing
    iv_df = (pd.DataFrame.from_dict(iv_values, orient="index", columns=["IV"])
             .sort_values("IV", ascending=False))
    plt.figure(figsize=(10, 8))
    sns.barplot(x=iv_df["IV"][:20], y=iv_df.index[:20], palette="viridis")
    plt.title("Top 20 Features by Information Value (IV)")
    plt.xlabel("Information Value")
    plt.ylabel("Features")
    plt.tight_layout()

    print(f"Features before FE : {X_train_clean.shape[1]}")
    print(f"Features after  FE : {X_train_fe.shape[1]}")
    print(f"\nTop WOE features by IV:")
    print(iv_df.head(10).to_string())

    # for next cells
    X_train_fe = X_train_fe
    X_test_fe = X_test_fe